## Breaking documents into chunks

In [ ]:
# Install required packages (run this cell once)
# Uncomment the line below and run if you get ModuleNotFoundError

# !pip install sentence-transformers langchain-chroma langchain-community langchain-openai chromadb


In [22]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
# Use langchain_community instead of langchain_huggingface to avoid version conflicts
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
# For accurate Llama tokenization (optional - requires HuggingFace auth)
try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Note: transformers not installed. Will use tiktoken approximation for Llama models.")

## We will divide our documents into chunks

## HuggingFace Authentication (for accurate Llama tokenization)

### ⚠️ Skip CLI Login - Use Environment Variable Instead

The CLI login is having issues. Use the `.env` file method instead (much easier!):

### Step-by-Step Process:

**1. Create a NEW Read Token:**
   - Go to: https://huggingface.co/settings/tokens
   - If you have an old token, delete it and create a new one
   - Click "New token"
   - Name it (e.g., "Llama Tokenizer")
   - Select **"Read"** token type
   - Click "Generate token"
   - **Copy the ENTIRE token** (starts with `hf_` and is very long)
   - ⚠️ Make sure you copy it completely - no truncation!

**2. Add Token to `.env` File:**
   - Open your `.env` file: `C:\Users\Cem\Desktop\projeler\YapayZekaUygulamalari\.env`
   - Add this line (replace with your actual token):
     ```
     HUGGINGFACE_TOKEN=hf_your_actual_token_here
     ```
   - ⚠️ **Important:**
     - No quotes around the token
     - No spaces before/after the `=`
     - Token must start with `hf_`
     - Copy the entire token (usually 40+ characters)

**3. Request Model Access:**
   - Go to: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
   - Click "Request access" button
   - Fill out the form
   - Wait for approval email (usually quick)

**4. Restart & Verify:**
   - **Restart your notebook kernel** (important!)
   - Run the verification cell below
   - You should see "✅ Token is valid"

**Note:** The code will automatically fall back to tiktoken approximation if authentication fails.


In [23]:
MODEL = "llama3.2"
db_name = "vector_db"

env_path = r"C:\Users\Cem\Desktop\projeler\YapayZekaUygulamalari\.env"
load_dotenv(env_path)

# HuggingFace token (optional - for accurate Llama tokenization)
# Priority: 1. Cached CLI token (most reliable), 2. Environment variable, 3. None (use approximation)
env_token = os.getenv("HUGGINGFACE_TOKEN", None)
cached_token = None

# Try to get cached token from CLI login first (most reliable)
try:
    from huggingface_hub import HfFolder
    cached_token = HfFolder.get_token()
except:
    pass

# Prioritize cached token if available (since CLI login worked)
if cached_token:
    HF_TOKEN = cached_token
    print("ℹ️  Using cached token from CLI login (hf auth login)")
elif env_token:
    HF_TOKEN = env_token
    print("ℹ️  Using token from environment variable (.env file)")
else:
    HF_TOKEN = None
    print("ℹ️  No token found - will use tiktoken approximation")

USE_HF_TOKENIZER = True  # Set to False to use tiktoken approximation instead

ℹ️  Using cached token from CLI login (hf auth login)


In [24]:
# Verify HuggingFace token and model access
print("=" * 60)
print("HuggingFace Token Verification")
print("=" * 60)

# Check which token source we're using
try:
    from huggingface_hub import HfFolder
    cached_token_check = HfFolder.get_token()
    env_token_check = os.getenv("HUGGINGFACE_TOKEN", None)
    
    if cached_token_check and HF_TOKEN == cached_token_check:
        token_source = "cached CLI login"
    elif env_token_check and HF_TOKEN == env_token_check:
        token_source = "environment variable"
    else:
        token_source = "unknown source"
except:
    token_source = "environment variable" if os.getenv("HUGGINGFACE_TOKEN") else "unknown source"

if HF_TOKEN:
    print(f"\n📋 Token found ({token_source})")
    print(f"   Length: {len(HF_TOKEN)} characters")
    print(f"   Starts with 'hf_': {HF_TOKEN.startswith('hf_')}")
    
    if not HF_TOKEN.startswith('hf_'):
        print(f"\n❌ ERROR: Token should start with 'hf_'")
        print(f"   Your token appears to be invalid. Please create a new token.")
    else:
        try:
            from huggingface_hub import whoami, HfApi
            api = HfApi(token=HF_TOKEN)
            
            # Check if token is valid
            print(f"\n🔍 Testing token validity...")
            user_info = whoami(token=HF_TOKEN)
            print(f"✅ Token is valid! Logged in as: {user_info.get('name', 'Unknown')}")
            
            # Check model access
            print(f"\n🔍 Checking model access...")
            try:
                model_info = api.model_info("meta-llama/Meta-Llama-3-8B-Instruct", token=HF_TOKEN)
                print(f"✅ Model access granted! You can use the accurate tokenizer.")
            except Exception as e:
                if "403" in str(e) or "gated" in str(e).lower():
                    print(f"⚠️  Model access not yet granted.")
                    print(f"   Please request access at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")
                    print(f"   Once approved, the tokenizer will work automatically.")
                else:
                    print(f"⚠️  Could not verify model access: {e}")
        except Exception as e:
            error_str = str(e)
            print(f"\n❌ Token verification failed!")
            if "400" in error_str or "Bad request" in error_str:
                print(f"   Error: Invalid token format or expired token")
                print(f"   Solution: Create a NEW token at https://huggingface.co/settings/tokens")
            elif "401" in error_str or "Unauthorized" in error_str:
                print(f"   Error: Token is invalid or revoked")
                print(f"   Solution: Create a NEW token and update your .env file")
            else:
                print(f"   Error: {e}")
            print(f"\n   Make sure your .env file has:")
            print(f"   HUGGINGFACE_TOKEN=hf_your_token_here")
            print(f"   (No quotes, no spaces around =)")
else:
    print("\n❌ No HUGGINGFACE_TOKEN found in environment.")
    print("\n   To fix this:")
    print("   1. Open your .env file")
    print("   2. Add: HUGGINGFACE_TOKEN=hf_your_token_here")
    print("   3. Restart your notebook kernel")
    print("   4. Run this cell again")

print("\n" + "=" * 60)


HuggingFace Token Verification

📋 Token found (cached CLI login)
   Length: 37 characters
   Starts with 'hf_': True

🔍 Testing token validity...
✅ Token is valid! Logged in as: Cemkaaa

🔍 Checking model access...
✅ Model access granted! You can use the accurate tokenizer.



In [25]:
knowledge_base_path = "rag_documents/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"found {len(files)} files in {knowledge_base_path}")

entire_knowledge_base = ""

for file_path in files:
     with open(file_path, 'r', encoding='utf-8') as file:
          entire_knowledge_base += file.read()
          entire_knowledge_base += "\n\n"
print(f"total number of characters in the knowledge base: {len(entire_knowledge_base):,}")

found 1041 files in rag_documents/**/*.md
total number of characters in the knowledge base: 401,572


## How many tokens in all of the document

In [26]:
# Token counting for different models
# For OpenAI models, use tiktoken directly
# For Llama models, try HuggingFace tokenizer first (more accurate), then fallback to cl100k_base

if MODEL.startswith("gpt") or MODEL in ["gpt-3.5-turbo", "gpt-4", "gpt-4-turbo"]:
    # Use tiktoken for OpenAI models
    try:
        encoding = tiktoken.encoding_for_model(MODEL)
        tokens = encoding.encode(entire_knowledge_base)
        token_count = len(tokens)
        print(f"Entire token count for {MODEL} (using tiktoken): {token_count:,}")
    except KeyError:
        # Fallback to cl100k_base encoding (GPT-4's encoding)
        encoding = tiktoken.get_encoding("cl100k_base")
        tokens = encoding.encode(entire_knowledge_base)
        token_count = len(tokens)
        print(f"Entire token count for {MODEL} (using cl100k_base): {token_count:,}")

elif MODEL.startswith("llama"):
    # Llama 3.2 has 128k vocab (vs GPT-4's 100k), so HuggingFace tokenizer is more accurate
    # But cl100k_base is a good approximation for English text
    
    if USE_HF_TOKENIZER and TRANSFORMERS_AVAILABLE:
        try:
            print(f"Loading HuggingFace tokenizer for {MODEL} (accurate tokenization)...")
            # Llama 3.2 uses the same tokenizer as Llama 3
            # If we have a token, use it; otherwise AutoTokenizer will use cached token automatically
            tokenizer_kwargs = {}
            if HF_TOKEN:
                tokenizer_kwargs["token"] = HF_TOKEN
            # If no token passed, AutoTokenizer will automatically use cached token from CLI login
            
            tokenizer = AutoTokenizer.from_pretrained(
                "meta-llama/Meta-Llama-3-8B-Instruct",
                **tokenizer_kwargs
            )
            tokens = tokenizer.encode(entire_knowledge_base, add_special_tokens=False)
            token_count = len(tokens)
            print(f"✅ Entire token count for {MODEL} (using HuggingFace tokenizer - ACCURATE): {token_count:,}")
            print(f"   Vocabulary size: {len(tokenizer.get_vocab()):,} tokens")
        except Exception as e:
            print(f"⚠️  HuggingFace tokenizer failed: {e}")
            print("   Falling back to tiktoken approximation...")
            # Fall through to tiktoken
            encoding = tiktoken.get_encoding("cl100k_base")
            tokens = encoding.encode(entire_knowledge_base)
            token_count = len(tokens)
            print(f"📊 Entire token count for {MODEL} (using cl100k_base approximation): {token_count:,}")
            print("   Note: This is close but not exact. Llama 3.2 has 128k vocab vs GPT-4's 100k.")
    else:
        # Use tiktoken approximation
        encoding = tiktoken.get_encoding("cl100k_base")
        tokens = encoding.encode(entire_knowledge_base)
        token_count = len(tokens)
        print(f"📊 Entire token count for {MODEL} (using cl100k_base approximation): {token_count:,}")
        print("   Note: Llama 3.2 has 128k vocab (vs GPT-4's 100k), so this is approximate.")
        print("   For accurate counts, set USE_HF_TOKENIZER=True and authenticate with HuggingFace.")
        print("   Instructions: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")

else:
    # For other models, use cl100k_base as a general approximation
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(entire_knowledge_base)
    token_count = len(tokens)
    print(f"Entire token count for {MODEL} (using cl100k_base approximation): {token_count:,}")

Loading HuggingFace tokenizer for llama3.2 (accurate tokenization)...


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

✅ Entire token count for llama3.2 (using HuggingFace tokenizer - ACCURATE): 169,184
   Vocabulary size: 128,256 tokens


## Dokümanları langchain objesi olarak kaydedelim

In [27]:
folders = glob.glob("rag_documents/*")

documents = []

for folder in folders:
     doc_type = os.path.basename(folder)
     loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
     folder_docs = loader.load()
     for doc in folder_docs:
          doc.metadata["doc_type"] = doc_type
          documents.append(doc)
print(f"loaded {len(documents)} documents")

loaded 1041 documents


In [28]:
documents[0]

Document(metadata={'source': 'rag_documents\\customers\\(101)TIGER_Lift_&_Handels_GmbH.md', 'doc_type': 'customers'}, page_content='---\ndoc_type: customer\ndoc_id: customer_101\ncustomer_id: 101\ncustomer_name: (101)TIGER Lift & Handels GmbH\nmarket_type: Yurtdisi\ndelivery_time: 14\n---\n\nMÜŞTERİ BİLGİSİ: (101)TIGER Lift & Handels GmbH\n\n- Müşteri ID: 101\n- Müşteri Adı: (101)TIGER Lift & Handels GmbH\n- Libal Müşteri Kodu: 101\n- Pazar: Yurtdisi\n- Teslimat Süresi: 14\n- Sevkiyat Günü: 5')

In [29]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"first chunk is \n\n{chunks[0]}")

Divided into 1041 chunks
first chunk is 

page_content='---
doc_type: customer
doc_id: customer_101
customer_id: 101
customer_name: (101)TIGER Lift & Handels GmbH
market_type: Yurtdisi
delivery_time: 14
---

MÜŞTERİ BİLGİSİ: (101)TIGER Lift & Handels GmbH

- Müşteri ID: 101
- Müşteri Adı: (101)TIGER Lift & Handels GmbH
- Libal Müşteri Kodu: 101
- Pazar: Yurtdisi
- Teslimat Süresi: 14
- Sevkiyat Günü: 5' metadata={'source': 'rag_documents\\customers\\(101)TIGER_Lift_&_Handels_GmbH.md', 'doc_type': 'customers'}


## Embedding Model Recommendations for Turkish RAG System

Based on your Turkish technical/manufacturing content, here are the best embedding models:

### 🏆 **Top Recommendation: `intfloat/multilingual-e5-large`**
- **Best quality** for Turkish + English multilingual content
- **1024 dimensions** - richer semantic representations
- Excellent for technical/domain-specific content
- **Slower** but highest accuracy
- **Use when**: Quality is most important, you can wait for embedding generation

### ⚡ **Best Balance: `intfloat/multilingual-e5-base`**
- **Good quality** with better speed than large
- **768 dimensions** - good balance
- Still excellent for Turkish content
- **Faster** than large version
- **Use when**: You want good quality with reasonable speed

### 🚀 **Fast Option: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`**
- **Better than your current MiniLM** (which you have in config)
- **768 dimensions**
- **Fastest** of the three
- Good for Turkish, but not as good as e5 models
- **Use when**: Speed is critical, you have many documents

### 📊 **Comparison Table**

| Model | Dimensions | Speed | Turkish Quality | Technical Content | Best For |
|-------|-----------|-------|----------------|-------------------|----------|
| `multilingual-e5-large` | 1024 | Slow | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Best quality |
| `multilingual-e5-base` | 768 | Medium | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Best balance |
| `paraphrase-multilingual-mpnet-base-v2` | 768 | Fast | ⭐⭐⭐⭐ | ⭐⭐⭐ | Fast & good |

**Recommendation for your 1,041 documents (169k tokens):**
- Start with **`intfloat/multilingual-e5-base`** - best balance for your use case
- If you need faster embedding, use `paraphrase-multilingual-mpnet-base-v2`
- If quality is critical, use `multilingual-e5-large`


In [32]:
# Choose your embedding model (uncomment one)
# RECOMMENDED: Best balance of quality and speed for Turkish content
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# Alternative options (uncomment to use):
# EMBEDDING_MODEL = "intfloat/multilingual-e5-base"  # Best balance, faster
# EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # Fast option
# EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # Your current config (fastest, lower quality)

print(f"📦 Selected embedding model: {EMBEDDING_MODEL}")
print(f"   This model will be used to create embeddings for your {len(chunks)} chunks")

# Auto-detect device (CPU or CUDA)
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"   Using device: {device}")

# Initialize the embedding model
# Note: First run will download the model (can take a few minutes)
print(f"\n⏳ Loading embedding model (this may take a minute on first run)...")
try:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': device},
        encode_kwargs={'normalize_embeddings': True}  # Normalize for better cosine similarity
    )
    print(f"✅ Embedding model loaded successfully!")
except Exception as e:
    if "tf_keras" in str(e) or "Keras" in str(e):
        print(f"❌ Keras compatibility error detected!")
        print(f"   Please run this command in a new cell:")
        print(f"   !pip install tf-keras")
        print(f"   Then restart kernel and try again.")
        raise
    else:
        raise


📦 Selected embedding model: intfloat/multilingual-e5-large
   This model will be used to create embeddings for your 1041 chunks
   Using device: cpu

⏳ Loading embedding model (this may take a minute on first run)...



modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

✅ Embedding model loaded successfully!


## Make vectors and store in chroma db

In [33]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"vector store created with {vector_store._collection.count()} documents")

vector store created with 1041 documents


In [ ]:
collection = vector_store._collection
count = collection.count()

sample_embeddings = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embeddings)
print(f"there are {count:,} vectors with {dimensions} dimensions in the vector store")

there are 1,041 vectors with 1024 in the vector store


In [35]:
result = collection.get(include=["documents", "embeddings", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'purple', 'orange', 'brown', 'pink', 'gray'][['customers', 'cutting_tools', 'machines', 'orders', 'processes', 'raw_materials', 'shifts', 'work_orders'].index(t)] for t in doc_types]

In [38]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [39]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [52]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import gradio as gr
from langchain_groq import ChatGroq

In [53]:
MODEL = "llama3.2"
MODEL_GROQ = "openai/gpt-oss-120b"  # or "llama-3.3-70b-versatile" for 70B model

load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

# Check if API key is loaded
if not groq_api_key:
    print("⚠️  WARNING: GROQ_API_KEY not found in environment variables!")
    print("   Please add it to your .env file: GROQ_API_KEY=your_key_here")
    print("   Get your key from: https://console.groq.com/keys")
else:
    print(f"✅ Groq API key loaded (starts with: {groq_api_key[:10]}...)")

retriever = vector_store.as_retriever()
llm = ChatOllama(model=MODEL, temperature=0.0)

# Initialize ChatGroq with API key
# Option 1: Pass API key explicitly (recommended)
llm_groq = ChatGroq(
    api_key=groq_api_key,  # Pass the API key here
    model=MODEL_GROQ, 
    temperature=0.0
)

# Option 2: If you don't pass api_key, ChatGroq will automatically 
# read from GROQ_API_KEY environment variable
# llm_groq = ChatGroq(model=MODEL_GROQ, temperature=0.0)

print(f"✅ LLM initialized: {MODEL} (Ollama)")
print(f"✅ Groq LLM initialized: {MODEL_GROQ}")

✅ Groq API key loaded (starts with: gsk_mKXvU6...)
✅ LLM initialized: llama3.2 (Ollama)
✅ Groq LLM initialized: openai/gpt-oss-120b


In [54]:
retriever.invoke("fabrikadaki hangi work order testere processini kullanıyor?")

[Document(id='0b448c05-0d0a-44b9-8958-499ace7b9e88', metadata={'source': 'rag_documents\\work_orders\\61,920-01-01-01.md', 'doc_type': 'work_orders'}, page_content='---\ndoc_type: work_order\ndoc_id: work_order_100109\nwork_order_id: 100109\nwork_order_code: 61,920-01-01-01\norder_no: 41060\ncustomer: (61)Danfoss Power Solutions II GmbH\nmachine_code: TS-5\nprocess: TESTERE\nquantity: 34.0\nproduced: 0.0\nwastage: 0.0\nstatus: Aktif\ncompletion_rate: 0.0\nconfirm_date: 2025-12-11T11\\:45\\:19\ndue_date: 2026-04-30\n---\n\nİŞ EMRİ: 61,920-01-01-01\n\nSipariş Bilgileri:\n- İş Emri ID: 100109\n- İş Emri Kodu: 61,920-01-01-01\n- Sipariş No: 41060\n- Müşteri Sipariş No: TEST1\n- Müşteri: (61)Danfoss Power Solutions II GmbH\n\nÜretim Bilgileri:\n- Makina: TS-5 - TESTERE\n- Proses: TESTERE\n- Sipariş Miktarı: 34 adet\n- Üretilen Miktar: 0 adet\n- Fire: 0 adet\n- Tamamlanma Oranı: %0.0\n- Durum: Aktif\n\nZaman Bilgileri:\n- Planlanan Ayar Süresi: 0.0 saat\n- Planlanan Üretim Süresi: 0 saniye/a

In [55]:
llm.invoke("fabrikadaki hangi work order testere processini kullanıyor?")

AIMessage(content="Fabrika içi Work Order Testere Processi, genellikle fabrika içi maintenance ve repair işlemlerinin yönetilmesinde kullanılan bir sistemdir. Bu sistem, fabrika içi makineleri ve equipment'leri restore etmek veya onarmak için kullanılan testere processini içerir.\n\nFabrika içi Work Order Testere Processi, genellikle fabrika içi maintenance ve repair işlemlerinin yönetilmesinde kullanılan bir sistemdir. Bu sistem, fabrika içi makineleri ve equipment'leri restore etmek veya onarmak için kullanılan testere processini içerir.\n\nBu processin amacı, fabrika içi makineleri ve equipment'leri restore etmek veya onarmak için needed olan work order'ları oluşturmak ve yönetmektedir. Fabrika içi Work Order Testere Processi, genellikle fabrika içi maintenance ve repair işlemlerinin yönetilmesinde kullanılan bir sistemdir.\n\nFabrika içi Work Order Testere Processi, genellikle fabrika içi makineleri ve equipment'leri restore etmek veya onarmak için needed olan work order'ları oluşt

In [56]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Libal Makina.
You are chatting with a user about Libal Makina.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [59]:
def answer_question(question: str, history):
     docs = retriever.invoke(question)
     context = "\n\n".join(doc.page_content for doc in docs)
     system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
     response = llm_groq.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
     return response.content

In [60]:
answer_question("which current work order is using the testere process?", [])

'The only active work order in the system that is assigned to the **TESTERE** (saw) process is:\n\n| Field | Value |\n|-------|-------|\n| **Work Order ID** | **100109** |\n| **Work Order Code** | **61,920-01-01-01** |\n| **Order No.** | 41060 |\n| **Customer** | (61) Danfoss Power Solutions II GmbH |\n| **Machine** | TS‑5 – TESTERE |\n| **Process** | **TESTERE** |\n| **Quantity Ordered** | 34\u202fadet |\n| **Produced** | 0\u202fadet |\n| **Status** | **Aktif** |\n| **Confirm Date** | 2025‑12‑11\u202f11:45:19 |\n| **Due Date** | 2026‑04‑30 |\n\nSo, **Work Order\u202f100109 (code\u202f61,920‑01‑01‑01)** is the current (active) work order that uses the **TESTERE** process.'

In [61]:
gr.ChatInterface(answer_question).launch()

c:\Users\Cem\Desktop\projeler\YapayZekaUygulamalari\KlasikMakineOgrenmesiTemelleri\MLFLOW\AWS\venv\lib\site-packages\gradio\chat_interface.py:347: UserWarning:

The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.



* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Rag Evaluation System

In [1]:
from evaluation.test import load_tests

In [2]:
tests = load_tests()

len(tests)

150

In [3]:
example = tests[0]
print(example.question)
print(example.reference_answer)
print(example.keywords)
print(example.category)

What is the machine code for CT-1?
CT-1 is a CNC TORNA machine with machine code CT-1.
['CT-1', 'machine', 'code']
direct_fact


In [4]:
from collections import Counter

count = Counter[str]([t.category for t in tests])
count

Counter({'direct_fact': 50,
         'temporal': 25,
         'numerical': 25,
         'relationship': 25,
         'spanning': 20,
         'holistic': 5})

In [5]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

GROQ API key found in the environment variables
Using the device cuda
Initializing vectorstore...
Creating retriever...
Pre-warming embedding model with a test query...
✅ Embedding model pre-warmed
✅ All components initialized


In [7]:
evaluate_retrieval(example)

  🔍 Retrieving context for: What is the machine code for CT-1?...
[fetch_context] Query: What is the machine code for CT-1?...
[fetch_context] Step 1: Encoding query with embedding model...
[fetch_context] Step 2: Query encoded (embedding dim: 1024)
[fetch_context] Step 3: Calling vectorstore.similarity_search() directly...
[fetch_context] Step 4: Retrieved 10 documents
  ✅ Retrieved 10 documents
  ✅ MRR: 1.0000, nDCG: 0.9218, Coverage: 100.0%


RetrievalEval(mrr=1.0, ndcg=0.9218108550740592, keywords_found=3, total_keywords=3, keyword_coverage=100.0)

In [7]:
eval1, answer, chunks = evaluate_answer(example)

[fetch_context] Query: 
What is the machine code for CT-1?...
[fetch_context] Calling retriever.invoke()...
[fetch_context] Retrieved 10 documents


In [8]:
eval1

AnswerEval(feedback='The answer correctly states that the machine code for CT-1 is CT-1, matching the reference. It includes all necessary information and the added note about the CNC TORNA machine is accurate and relevant, so the response is fully accurate, complete, and on point.', accuracy=5.0, completeness=5.0, relevance=5.0)

In [9]:
print(eval1.feedback)
print(eval1.accuracy)
print(eval1.completeness)
print(eval1.relevance)

The answer correctly states that the machine code for CT-1 is CT-1, matching the reference. It includes all necessary information and the added note about the CNC TORNA machine is accurate and relevant, so the response is fully accurate, complete, and on point.
5.0
5.0
5.0
